# Notebook 03: Preamble Extraction Proposal

## Background and Objectives
This notebook implements the Task 3 preamble extraction proposal across all 193 unique national constitutions.

Following the diagnostic findings of Notebook 02—which revealed that 36 constitutions in the dataset contain full-body legislative articles rather than genuine preambles, and identified HTML entity corruption across multiple records—this pipeline establishes a transparent, rule-based extraction methodology:
1. **Entity and Whitespace Normalization**: Decodes HTML entities (`html.unescape`), resolves unescaped/broken entity remnants (`x27;`, `quot;`, `nbsp;`, `amp;`, stray `gt;`/`lt;`), and collapses multiple spaces and newlines into a single space.
2. **Rule-Based Extraction & Status Assignment**:
   - `no_preamble_in_source`: Constitutions lacking a preamble in the source text (starting immediately with structural headings like Chapter/Article/Part and exceeding 1,000 words). The proposal sets `extracted_text = ""` (0 words) to avoid feeding body text into preamble models.
   - `long_cut`: Constitutions where a genuine preamble is followed immediately by the constitution body (exceeding 1,000 words without a structural start). We locate the first structural heading after word 30 and isolate the preamble.
   - `short_keep`: Standard preambles (<= 400 words, non-structural start). Kept intact.
   - `medium_review`: Preambles between 401 and 1,000 words without structural start. Kept intact for review.
   - `manual_review`: Borderline cases requiring individual inspection (e.g., structural starts under 1,000 words, or long texts without a heading after word 30).
3. **Audit Reports**: Detailed breakdowns per status, crosstabulations against original cluster assignments, cut boundaries, and extracted texts under 30 words.
4. **Data Deliverable**: Generating `data/processed/preamble_extraction_proposal.csv` with columns: `Country, orig_cluster, orig_words, status, rule, extracted_words, extracted_text`.

In [1]:
import os
import re
import html
import json
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

data_dir = '../data/processed'
clustered_csv_path = os.path.join(data_dir, 'constitution_preamble_clustered.csv')
cleaned_csv_path = os.path.join(data_dir, 'constitution_dataset_cleaned.csv')
proposal_csv_path = os.path.join(data_dir, 'preamble_extraction_proposal.csv')

print(f"Data directory: {os.path.abspath(data_dir)}")


Data directory: /home/devpanchal/Projects/PreambleMap/data/processed


## 1. Data Ingestion and Dataset Alignment

We load `data/processed/constitution_preamble_clustered.csv` (which contains original cluster assignments) and `data/processed/constitution_dataset_cleaned.csv` (which contains raw Preamble text).
- **Cyprus**: Duplicate row 192 (an identical duplicate of row 44) is dropped, yielding exactly 193 unique countries.
- **Slovakia**: The missing raw text (`NaN`) is populated in-memory from Notebook 01 Cell 31 (`slovak_republic` string).

In [2]:
df_clust = pd.read_csv(clustered_csv_path)
df_clean = pd.read_csv(cleaned_csv_path)

# Drop Cyprus duplicate row 192
df = df_clust.drop(index=192).reset_index(drop=True)
df['Preamble'] = df_clean.drop(index=192).reset_index(drop=True)['Preamble']

# Load Slovakia text from Notebook 01 Cell 31
with open('01_data_preprocessing.ipynb', 'r', encoding='utf-8') as f:
    nb01 = json.load(f)
loc = {}
exec(''.join(nb01['cells'][31]['source']), {'df': pd.DataFrame([{'Country': 'Slovakia', 'Preamble': None}])}, loc)
df.loc[df['Country'] == 'Slovakia', 'Preamble'] = loc['slovak_republic']

print(f"Total rows ingested: {len(df)} countries")
print(f"Cyprus occurrences: {len(df[df['Country'] == 'Cyprus'])}")
print(f"Slovakia Preamble length (chars): {len(df.loc[df['Country'] == 'Slovakia', 'Preamble'].values[0])}")


Total rows ingested: 193 countries
Cyprus occurrences: 1
Slovakia Preamble length (chars): 120730


## 2. Step 1: Entity Cleanup and Normalization

Before assigning status or performing cuts, we apply entity and whitespace normalization across all 193 rows:
1. `html.unescape` decodes standard HTML entities.
2. Unescaped/broken entity remnants are repaired:
   - `x27;` -> `'` (apostrophe)
   - `quot;` -> `"` (quotation mark)
   - `nbsp;` -> ` ` (space)
   - `amp;` -> `&` (ampersand)
   - Stray `gt;` and `lt;` are removed using word boundaries (`\bgt;`, `\blt;`) so words like *guilt;*, *silt;*, and *salt;* are preserved.
3. Whitespace characters and newlines are collapsed to single spaces (`\s+` -> `' '`) and stripped.

In [3]:
def clean_entities(text):
    if pd.isna(text):
        return ''
    t = html.unescape(str(text))
    t = t.replace('x27;', "'").replace('quot;', '"').replace('nbsp;', ' ').replace('amp;', '&')
    t = re.sub(r'\bgt;', ' ', t)
    t = re.sub(r'\blt;', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

raw_texts = df['Preamble'].tolist()
cleaned_texts = [clean_entities(t) for t in raw_texts]
df['cleaned_raw'] = cleaned_texts
df['orig_words'] = df['cleaned_raw'].apply(lambda x: len(x.split()))
df['orig_cluster'] = df['cluster']

changed_rows = sum(1 for r, c in zip(raw_texts, cleaned_texts) if str(r) != c)
print(f"Total rows modified by Step 1 entity/whitespace cleanup: {changed_rows}")

print("\n=== 5 BEFORE / AFTER ENTITY CLEANUP EXAMPLES (WITH CONTEXT) ===")
sample_countries = ['Afghanistan', 'Andorra', 'Bangladesh', 'Central African Rep.', 'New Zealand']
for sc in sample_countries:
    raw_val = df.loc[df['Country'] == sc, 'Preamble'].values[0]
    cl_val = df.loc[df['Country'] == sc, 'cleaned_raw'].values[0]
    
    if sc == 'Afghanistan':
        pat = 'x27;'
    elif sc == 'Andorra':
        pat = 'quot;'
    elif sc == 'Bangladesh':
        pat = 'x27;'
    elif sc == 'Central African Rep.':
        pat = 'quot;'
    elif sc == 'New Zealand':
        pat = 'nbsp;'
    
    idx = raw_val.find(pat)
    start = max(0, idx - 30)
    end_raw = min(len(raw_val), idx + len(pat) + 30)
    snippet_before = raw_val[start:end_raw]
    
    prefix = raw_val[max(0, idx - 20):idx]
    cl_idx = cl_val.find(prefix)
    snippet_after = cl_val[max(0, cl_idx - 10):cl_idx + len(prefix) + 35]
    
    print(f"\nCountry: {sc} (Target Entity: {pat})")
    print(f"  BEFORE: {repr(snippet_before)}")
    print(f"  AFTER:  {repr(snippet_after)}")


Total rows modified by Step 1 entity/whitespace cleanup: 67

=== 5 BEFORE / AFTER ENTITY CLEANUP EXAMPLES (WITH CONTEXT) ===

Country: Afghanistan (Target Entity: x27;)
  BEFORE: ' of the martyrs of the countryx27;s freedom; Comprehending that '
  AFTER:  " of the martyrs of the country's freedom; Comprehending that a un"

Country: Andorra (Target Entity: quot;)
  BEFORE: 'ions, Desiring that the motto quot;virtus, unita, fortiorquot;, w'
  AFTER:  'ions, Desiring that the motto "virtus, unita, fortior", which has'

Country: Bangladesh (Target Entity: x27;)
  BEFORE: ' independent, sovereign Peoplex27;s Republic of Bangladesh; Pled'
  AFTER:  " independent, sovereign People's Republic of Bangladesh; Pledging"

Country: Central African Rep. (Target Entity: quot;)
  BEFORE: ' respect for the principle of quot;Zo Kwe Zoquot; declared by the'
  AFTER:  ' respect for the principle of "Zo Kwe Zo" declared by the Foundin'

Country: New Zealand (Target Entity: nbsp;)
  BEFORE: 'Legislature Act

## 3. Step 2: Status Assignment Heuristic

We evaluate each country using the extended `starts_with_structure` heuristic and word count bands:

1. `starts_with_structure`: Checks the first 12 words of the text for:
   - Structural keywords: `Part`, `Chapter`, `Title`, `Article`, `Section`, `Book`, `Schedule`, `Preliminary`.
   - Roman numerals: `I`, `II`, `III`, `IV`, etc.
   - Extended multi-word structural headings: `Fundamental Principles`, `Basic Law`, `General Provisions`, `Instrument of Government`, `Constitution Act`.

2. **Precedence Hierarchy**:
   1. `no_preamble_in_source`: `starts_with_structure is True` and `words > 1000`. Source text contains only constitution body articles. `extracted_text = ""` and `extracted_words = 0`.
   2. `long_cut`: `not starts_with_structure` and `words > 1000`. Locates the first structural heading (`Chapter`, `Title`, `Part`, `Article`, `Art`, `Section`, with optional numeral/Roman numeral) occurring AFTER word 30. The text is cut immediately before this heading. If none is found or the extract would be under 30 words, status is set to `manual_review`.
   3. `short_keep`: `not starts_with_structure` and `words <= 400`. Kept intact.
   4. `medium_review`: `not starts_with_structure` and `401 <= words <= 1000`. Kept intact for review.
   5. `manual_review`: Everything else (e.g. structural start with words <= 1000, or long texts without cut heading).

The resulting extraction proposal is saved to `data/processed/preamble_extraction_proposal.csv`.

In [4]:
extended_phrases = [
    'fundamental principles', 'basic law', 'general provisions',
    'instrument of government', 'constitution act'
]
single_keywords = {
    'part', 'chapter', 'title', 'article', 'section', 'book', 'schedule', 'preliminary'
}
roman_numerals = {'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii'}

def check_structure_start(text):
    words = text.split()
    first_12 = words[:12]
    first_12_str = ' '.join(first_12).lower()
    has_phrase = any(phrase in first_12_str for phrase in extended_phrases)
    has_kw = any(re.sub(r'^[^\w]+|[^\w]+$', '', w).lower() in single_keywords for w in first_12)
    has_rm = any(re.sub(r'^[^\w]+|[^\w]+$', '', w).lower() in roman_numerals for w in first_12)
    return has_phrase or has_kw or has_rm

numeral_or_roman = r'(?:[0-9]+|[IVXLCDMivxlcdm]+|One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten|First|Second|Third|Fourth|Fifth)'
heading_re = re.compile(
    r'\b(Chapter|Title|Part|Article|Art|Section)(?:\s+' + numeral_or_roman + r'\b|\b(?!\s+(?:of|in|and|the|a|an|to|for|with|by|from|or|meant)\b))',
    re.IGNORECASE
)

status_list = []
rule_list = []
extracted_text_list = []
extracted_words_list = []

for i, r in df.iterrows():
    txt = r['cleaned_raw']
    words = txt.split()
    n_words = len(words)
    is_struct = check_structure_start(txt)
    
    if is_struct and n_words > 1000:
        st = 'no_preamble_in_source'
        rl = 'structural_start_and_words_gt_1000'
        ext_txt = ''
        ext_w = 0
    elif (not is_struct) and n_words > 1000:
        prefix_30 = ' '.join(words[:30])
        rest = ' '.join(words[30:])
        m = heading_re.search(rest)
        if m:
            cut_pos = len(prefix_30) + 1 + m.start()
            ext = txt[:cut_pos].strip()
            ext_words = len(ext.split())
            if ext_words >= 30:
                st = 'long_cut'
                rl = f'cut_before_{m.group(0).replace(" ", "_")}'
                ext_txt = ext
                ext_w = ext_words
            else:
                st = 'manual_review'
                rl = 'extract_under_30_words'
                ext_txt = txt
                ext_w = n_words
        else:
            st = 'manual_review'
            rl = 'no_heading_found_after_word_30'
            ext_txt = txt
            ext_w = n_words
    elif (not is_struct) and n_words <= 400:
        st = 'short_keep'
        rl = 'non_structural_words_le_400'
        ext_txt = txt
        ext_w = n_words
    elif (not is_struct) and (401 <= n_words <= 1000):
        st = 'medium_review'
        rl = 'non_structural_words_401_to_1000'
        ext_txt = txt
        ext_w = n_words
    else:
        st = 'manual_review'
        rl = 'structural_start_words_le_1000'
        ext_txt = txt
        ext_w = n_words
        
    status_list.append(st)
    rule_list.append(rl)
    extracted_text_list.append(ext_txt)
    extracted_words_list.append(ext_w)

df['status'] = status_list
df['rule'] = rule_list
df['extracted_text'] = extracted_text_list
df['extracted_words'] = extracted_words_list

# Save proposal CSV
proposal_cols = ['Country', 'orig_cluster', 'orig_words', 'status', 'rule', 'extracted_words', 'extracted_text']
df[proposal_cols].to_csv(proposal_csv_path, index=False)
print(f"Saved proposal CSV to: {proposal_csv_path}")
print(f"Total rows in proposal: {len(df)}")


Saved proposal CSV to: ../data/processed/preamble_extraction_proposal.csv
Total rows in proposal: 193


## 4. Step 3: Diagnostic Reports

We generate detailed reports inspecting status distributions, cut boundaries, review candidates, and short preambles:
- **3a**: Status value counts and status by original cluster crosstab.
- **3b**: `long_cut` cut boundaries (120 characters before and 120 characters after cut).
- **3c**: `medium_review` countries (first 120 and last 150 characters).
- **3d**: `manual_review` countries (first 150 and last 150 characters).
- **3e**: `no_preamble_in_source` countries (Country, orig_words, orig_cluster).
- **3f**: Preambles with extracted_words < 30.

In [5]:
print("=== REPORT 3A: STATUS VALUE COUNTS ===")
status_counts = df['status'].value_counts()
display(pd.DataFrame({'Count': status_counts, 'Percentage': (status_counts / len(df) * 100).round(2)}))

print("\n=== REPORT 3A: CROSSTAB OF STATUS BY ORIGINAL CLUSTER ===")
crosstab_df = pd.crosstab(df['status'], df['orig_cluster'], margins=True)
display(crosstab_df)


=== REPORT 3A: STATUS VALUE COUNTS ===


,Count,Percentage
status,,
short_keep,116,60.10
no_preamble_in_source,30,15.54
medium_review,29,15.03
manual_review,12,6.22
long_cut,6,3.11



=== REPORT 3A: CROSSTAB OF STATUS BY ORIGINAL CLUSTER ===


orig_cluster,0,1,2,3,4,5,6,All
status,,,,,,,,
long_cut,0,0,2,1,3,0,0,6
manual_review,0,2,5,1,2,2,0,12
medium_review,1,0,5,6,8,1,8,29
no_preamble_in_source,0,0,7,0,11,12,0,30
short_keep,25,15,6,12,17,13,28,116
All,26,17,25,20,41,28,36,193


In [6]:
print("=== REPORT 3B: LONG_CUT BOUNDARY INSPECTION ===")
long_cut_df = df[df['status'] == 'long_cut'].sort_values('Country')

for _, r in long_cut_df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    ext = r['extracted_text']
    cut_pos = len(ext)
    after_slice = txt[cut_pos:].lstrip()
    
    before_cut = ext[-120:] if len(ext) >= 120 else ext
    after_cut = after_slice[:120]
    
    print(f"\nCountry: {c}")
    print(f"  Original words:  {r['orig_words']}")
    print(f"  Extracted words: {r['extracted_words']}")
    print(f"  Rule applied:    {r['rule']}")
    print(f"  120 chars BEFORE cut: {repr(before_cut)}")
    print(f"  120 chars AFTER cut:  {repr(after_cut)}")


=== REPORT 3B: LONG_CUT BOUNDARY INSPECTION ===

Country: France
  Original words:  13608
  Extracted words: 102
  Rule applied:    cut_before_ARTICLE
  120 chars BEFORE cut: 'd on the common ideal of liberty, equality and fraternity and conceived for the purpose of their democratic development.'
  120 chars AFTER cut:  'ARTICLE France shall be an indivisible, secular, democratic and social Republic. It shall ensure the equality of all cit'

Country: Libya
  Original words:  2952
  Extracted words: 211
  Rule applied:    cut_before_CHAPTER_ONE
  120 chars BEFORE cut: 'ercise of power in the transitional period until the adoption of a permanent Constitution by general popular referendum.'
  120 chars AFTER cut:  'CHAPTER ONE. GENERAL PROVISIONS Article Libya shall be an independent democratic state in which the people shall be the '

Country: New Zealand
  Original words:  3310
  Extracted words: 211
  Rule applied:    cut_before_Part
  120 chars BEFORE cut: 'Parts, as follows:Division 

In [7]:
print("=== REPORT 3C: MEDIUM_REVIEW (401 TO 1000 WORDS) ===")
med_df = df[df['status'] == 'medium_review'].sort_values('Country')
print(f"Total medium_review countries: {len(med_df)}")

for _, r in med_df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    w = r['orig_words']
    first_120 = txt[:120]
    last_150 = txt[-150:]
    print(f"\nCountry: {c} | Words: {w} | Cluster: {r['orig_cluster']}")
    print(f"  First 120 chars: {repr(first_120)}")
    print(f"  Last 150 chars:  {repr(last_150)}")


=== REPORT 3C: MEDIUM_REVIEW (401 TO 1000 WORDS) ===
Total medium_review countries: 29

Country: Angola | Words: 580 | Cluster: 4
  First 120 chars: 'We, the people of Angola, through its lawful representatives, the legislators of the nation, freely elected in the parli'
  Last 150 chars:  'ure generations and to the exercise of our sovereignty; We hereby pass this Constitution as the Supreme and Fundamental Law of the Republic of Angola.'

Country: Bahrain | Words: 742 | Cluster: 2
  First 120 chars: 'In the name of God, the Merciful, the Compassionate Constitution of the Kingdom of Bahrain Foreword to the Constitution '
  Last 150 chars:  'al Action Charter and that complement all the unamended texts. We have attached an explanation memorandum which will be used to explain its judgement.'

Country: Belize | Words: 486 | Cluster: 4
  First 120 chars: 'WHEREAS the people of Belize-affirm that the Nation of Belize shall be founded upon principles which acknowledge the sup'
  Last 150 ch

In [8]:
print("=== REPORT 3D: MANUAL_REVIEW COUNTRIES ===")
manual_df = df[df['status'] == 'manual_review'].sort_values('Country')
print(f"Total manual_review countries: {len(manual_df)}")

for _, r in manual_df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    w = r['orig_words']
    first_150 = txt[:150]
    last_150 = txt[-150:]
    print(f"\nCountry: {c} | Words: {w} | Cluster: {r['orig_cluster']} | Rule: {r['rule']}")
    print(f"  First 150 chars: {repr(first_150)}")
    print(f"  Last 150 chars:  {repr(last_150)}")


=== REPORT 3D: MANUAL_REVIEW COUNTRIES ===
Total manual_review countries: 12

Country: Algeria | Words: 1307 | Cluster: 2 | Rule: no_heading_found_after_word_30
  First 150 chars: 'The Algerian people are a free people; and they are resolved to remain so. Stretching back over thousands of years, their history is marked by a progr'
  Last 150 chars:  'uture generations, the successors of the pioneers of freedom and the founders of the free society. This shall be part and parcel of this Constitution.'

Country: Armenia | Words: 78 | Cluster: 1 | Rule: structural_start_words_le_1000
  First 150 chars: 'The Armenian People, accepting as a basis the fundamental principles of Armenian statehood and pan-national aspirations enshrined in the Declaration o'
  Last 150 chars:  '-being, and civic solidarity of the generations, and affirming its commitment to universal values, adopts the Constitution of the Republic of Armenia.'

Country: China | Words: 1160 | Cluster: 3 | Rule: no_heading_found_a

In [9]:
print("=== REPORT 3E: NO_PREAMBLE_IN_SOURCE COUNTRIES ===")
no_preamble_df = df[df['status'] == 'no_preamble_in_source'][['Country', 'orig_words', 'orig_cluster']].sort_values('Country')
print(f"Total no_preamble_in_source countries: {len(no_preamble_df)}")
display(no_preamble_df)


=== REPORT 3E: NO_PREAMBLE_IN_SOURCE COUNTRIES ===
Total no_preamble_in_source countries: 30


,Country,orig_words,orig_cluster
8,Australia,16303,4
9,Austria,47164,5
22,Botswana,30727,4
30,Canada,20145,4
44,Cyprus,34466,5
46,Denmark,6023,5
58,Finland,12987,5
80,Israel,17607,2
81,Italy,11305,5
82,Jamaica,36267,4


In [10]:
print("=== REPORT 3F: EXTRACTED WORDS < 30 ===")
short_extracted_df = df[df['extracted_words'] < 30].sort_values('extracted_words', ascending=False)

non_empty_short = short_extracted_df[short_extracted_df['extracted_words'] > 0]
print(f"1. Short Preambles with Non-Empty Text (0 < words < 30): {len(non_empty_short)} countries")
for _, r in non_empty_short.iterrows():
    print(f"\nCountry: {r['Country']} | Words: {r['extracted_words']} | Status: {r['status']} | Cluster: {r['orig_cluster']}")
    print(f"  Full Text: {repr(r['extracted_text'])}")

empty_short = short_extracted_df[short_extracted_df['extracted_words'] == 0]
print(f"\n2. Constitutions with No Preamble in Source (words = 0, empty text): {len(empty_short)} countries")
print(f"Countries: {', '.join(empty_short['Country'].tolist())}")


=== REPORT 3F: EXTRACTED WORDS < 30 ===
1. Short Preambles with Non-Empty Text (0 < words < 30): 3 countries

Country: Latvia | Words: 15 | Status: short_keep | Cluster: 1
  Full Text: 'The people of Latvia, in freely elected Constitutional Assembly, have adopted the following State Constitution:'

Country: Greece | Words: 11 | Status: short_keep | Cluster: 0
  Full Text: 'In the name of the Holy and Consubstantial and Indivisible Trinity'

Country: Belgium | Words: 11 | Status: short_keep | Cluster: 5
  Full Text: 'In the name of the Belgian people, the National Congress decrees:'

2. Constitutions with No Preamble in Source (words = 0, empty text): 30 countries
Countries: Australia, Botswana, Canada, Cyprus, Denmark, Austria, Finland, Israel, Italy, Jamaica, Lesotho, Luxembourg, Malaysia, Maldives, Malta, Mauritius, Mexico, Monaco, Netherlands, Norway, Oman, Qatar, Romania, Saudi Arabia, Sierra Leone, Singapore, Somalia, Sweden, Uruguay, Yemen
